# N2 Physics Validation Review

This notebook is for human physics validation of the package outputs that
support Notebook 2. It is not an automated pass/fail test. The goal is to
make the generated physics visible enough for manual review before package
changes are treated as scientifically acceptable.

The active package is imported as `lrom`. The Notebook 1 snapshot is imported
as `lrom_N1` so changes can be compared against the earlier milestone when
that is useful.

## Review Rules

Use this notebook to inspect figures, tables, parameter ranges, and method
comparisons. Automated tests may check that arrays exist and have the right
dimensions, but they should not decide whether the physics is acceptable.

Manual review notes should be written directly in markdown cells below the
relevant plot/table when you inspect an executed copy of this notebook.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "lrom").is_dir()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

for name in list(sys.modules):
    if name == "lrom" or name.startswith("lrom."):
        del sys.modules[name]
    if name == "lrom_legacy" or name.startswith("lrom_legacy."):
        del sys.modules[name]

import lrom
import lrom_legacy.N1 as lrom_N1

plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

print("repo root:", ROOT)
print("active package:", lrom.__version__, lrom.LROM.__module__)
print("Notebook 1 snapshot:", lrom_N1.__version__, lrom_N1.LROM.__module__)

## Validation Case

The default validation case matches the Notebook 2 scaffold:
$^{40}$Ca$(n,n)$ at $E_{lab}=14.1$ MeV, with the future cross-section work
assembled from multiple partial waves.

In [ ]:
validation_case = {
    "target": (40, 20),
    "projectile": (1, 0),
    "lab_energy": 14.1,
    "channels": tuple(range(0, 11)),
    "angles_degrees": np.linspace(1.0, 179.0, 120),
}

pd.Series(
    {
        "target": "40Ca",
        "projectile": "neutron",
        "E_lab [MeV]": validation_case["lab_energy"],
        "channels": f"{validation_case['channels'][0]} ... {validation_case['channels'][-1]}",
        "angle count": len(validation_case["angles_degrees"]),
    },
    name="manual validation case",
)

## Optical-Potential Parameter Review

The table below should be checked before trusting any sampled outputs. Confirm
parameter names, central values, units, and roles match the intended ROSE-paper
optical potential convention.

In [ ]:
optical_parameters = pd.DataFrame(
    [
        ("Vv", 46.7238, "MeV", "real volume depth"),
        ("Wv", 1.72334, "MeV", "imaginary volume depth"),
        ("Wd", -7.2357, "MeV", "imaginary surface depth"),
        ("Vso", 6.1, "MeV", "real spin-orbit strength"),
        ("Rv", 4.0538, "fm", "real volume radius"),
        ("Rd", 4.4055, "fm", "surface radius"),
        ("Rso", 1.01 * 40 ** (1.0 / 3.0), "fm", "spin-orbit radius"),
        ("av", 0.6718, "fm", "real volume diffuseness"),
        ("ad", 0.5379, "fm", "surface diffuseness"),
        ("aso", 0.60, "fm", "spin-orbit diffuseness"),
    ],
    columns=["parameter", "central", "unit", "role"],
)
optical_parameters

**Manual review note:** confirm whether these values and signs are the ones
you want the package to own for Notebook 2.

## Current Package Smoke Review

This section uses the current package on a small wavefunction-level case. It
is not the final cross-section validation. It simply confirms that active
`lrom` and the frozen `lrom_N1` snapshot can be run side-by-side when needed.

In [ ]:
RUN_SMALL_WAVEFUNCTION_REVIEW = False

if RUN_SMALL_WAVEFUNCTION_REVIEW:
    active = lrom.LROM(
        target=(40, 20),
        projectile=(1, 0),
        lab_energy=14.1,
        l=0,
        potential="ws_1",
    )
    legacy = lrom_N1.LROM(
        target=(40, 20),
        projectile=(1, 0),
        lab_energy=14.1,
        l=0,
        potential="ws_1",
    )

    Vv0 = active.central_parameters["Vv"]
    ranges = {"Vv": (0.95 * Vv0, 1.05 * Vv0)}
    for model in (active, legacy):
        model.sampling(
            training_ranges=ranges,
            testing_ranges=ranges,
            training_size=5,
            testing_size=3,
            strategy="linspace",
            mesh_size=96,
            eim_basis_size=4,
        )
        model.train(basis_size=2, predictor="parameters", predictor_count=1)

    case_id = active.samples.design.testing.case_ids[1]
    active_case = active.testing_case(case_id=case_id)
    legacy_case = legacy.testing_case(case_id=case_id)

    fig, ax = plt.subplots(figsize=(7.2, 4.0))
    ax.plot(active_case.radius, np.real(active_case.high_fidelity[0]), color="black", label="active FOM")
    ax.plot(active_case.radius, np.real(active_case.lrom[0]), color="tab:orange", label="active LROM")
    ax.plot(legacy_case.radius, np.real(legacy_case.lrom[0]), "--", color="tab:blue", label="N1 LROM")
    ax.set_xlabel("r [fm]")
    ax.set_ylabel("Re(phi)")
    ax.set_title(f"Small side-by-side smoke review: {case_id}")
    ax.legend()
    plt.show()
else:
    print("Set RUN_SMALL_WAVEFUNCTION_REVIEW = True to run this optional smoke review.")

**Manual review note:** when this optional section is run, inspect whether the
active and `N1` wavefunction curves differ for an expected reason. This cell
should not become an automated physics acceptance test.

## Future N2 Cross-Section Review

Once the package owns cross-section assembly, this section should become the
main review. The expected outputs are visible tables and plots: representative
cross sections, train/test error distributions, CAT plots, potential rainbows,
and predictor-location overlays.

In [ ]:
future_cross_section_review = """
xs_emulator = lrom.LROM(
    target=(40, 20),
    projectile=(1, 0),
    lab_energy=14.1,
    l=tuple(range(0, 11)),
    potential="full_woods-saxon",
)
xs_emulator.sampling(...)
xs_emulator.train_cross_section_models(...)
xs_results = xs_emulator.evaluate_cross_sections(...)

# Human-review figures:
# 1. FOM / LS floor / ROSE ROM / predictor LROM cross-section curves.
# 2. Train/test split error distributions.
# 3. CAT plot over basis and operator sizes.
# 4. Potential rainbows and predictor locations for selected channels.
""".strip()

print(future_cross_section_review)

## Manual Review Checklist

- Parameter names and central optical-potential values are correct.
- Training and testing ranges are physically meaningful.
- ROSE and LROM consume identical parameter rows.
- Cross-section curves have the expected angular structure and scale.
- LS floor behaves like a lower-bound diagnostic, not a competing method claim.
- Error distributions separate train/test behavior clearly.
- CAT timing includes online prediction only.
- Predictor locations are physically interpretable.
- Any active-vs-`N1` difference is explained in notebook text before acceptance.